# Breast Cancer Wisconsin (Diagnostic) – Tüm Deneyler (Hakem revizyonu)

**Çalıştırma:** Çalışma Zamanı → Tümünü çalıştır. Fraud deneylerinin yapıldığı çalışma zamanı türüyle (CPU/GPU) aynı türü seçin.
Toplam süre Colab CPU'da yaklaşık 1–1,5 saattir. En uzun adımlar: kuantum sadakat matrisleri (~20–25 dk) ve 8 kübitlik QNN (~20 dk). Sekmeyi açık tutun.

**Bittiğinde:** Son hücre `bc_sonuclar.zip` dosyasını indirir (tüm ham sonuçlar JSON + grafikler + bölme indeksleri). Bu zip dosyasını gönderin.

| Hücre | İçerik | İlgili hakem yorumu |
|---|---|---|
| 1 | Ortam bilgisi, veri, 5-katlı bölme, sızıntısız ön-işleme, PCA varyansı | H1-Y12, H2-Y2, Y3, Y10, Y19-V |
| 2 | Kuantum sadakat matrisleri (Adjoint 4 kübit, Swap Test 9 kübit) | H2-Y9, Y10 |
| 3 | Klasik SVM / QSVM, Klasik k-NN / QKNN | H2-Y4 |
| 4 | K-Means, Kuantum Çekirdek Kümeleme, Swap-Test Q-Means (Hungarian eşleme) | H2-Y13 |
| 5 | Saf ANN, parametre eşlemeli klasik kontrol, QNN 2/4/8 kübit; eşit bütçe, öğrenme eğrileri | H2-Y5, Y8, Y11, Y19-VI |
| 6 | Klasik ANN çökme analizi: 5 tohum × 3 öğrenme oranı, ölü ReLU tespiti | H2-Y6 |
| 7 | Klasik AE ve QAE anomali tespiti (normal/anomali tanımı, eşik, PR-AUC) | H2-Y12 |
| 8 | Tüm sonuçları kaydet ve indir | H2-Y18 |


In [ ]:
!pip install -q pennylane

In [ ]:
# ======================================================================================
# BREAST CANCER WISCONSIN (DIAGNOSTIC) – TÜM DENEYLER TEK DEFTER
# Hakem yorumlarına göre: tabakalı 5-katlı CV (aynı katmanlar), sızıntısız ön-işleme, mean ± SD,
# eşit eğitim bütçesi + öğrenme eğrileri, parametre-eşlemeli klasik kontrol, çoklu tohum ANN analizi,
# Hungarian eşleme, ayrıntılı anomali protokolü, PR-AUC, ortam bilgisi, ham sonuç ve bölme indeksi kaydı.
# ======================================================================================
import json, os, platform, sys, time, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pennylane as qml
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from scipy.optimize import linear_sum_assignment
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix,
                             adjusted_rand_score, normalized_mutual_info_score, silhouette_score)
warnings.filterwarnings("ignore")
plt.rcParams.update({"font.size": 10, "font.family": "serif"})

OUT = "bc_sonuclar"; os.makedirs(OUT, exist_ok=True)
ALL = {}   # tüm ham sonuçlar burada toplanır ve sonda tek JSON olarak kaydedilir

def ms(a, scale=1.0, nd=2):
    a = np.asarray(a, float) * scale
    return f"{a.mean():.{nd}f} ± {a.std():.{nd}f}"

# ---------------- 0. ORTAM BİLGİSİ (Hakem #1 Yorum #12, Hakem #2 Yorum #10) ----------------
def environment():
    import sklearn, scipy
    info = {"Python": sys.version.split()[0], "OS": platform.platform(), "NumPy": np.__version__,
            "scikit-learn": sklearn.__version__, "SciPy": scipy.__version__, "PennyLane": qml.__version__,
            "PyTorch": torch.__version__, "GPU": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Yok (CPU)",
            "CPU çekirdek": os.cpu_count()}
    try:
        info["RAM (GB)"] = round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1)
        with open("/proc/cpuinfo") as f:
            for line in f:
                if "model name" in line: info["CPU modeli"] = line.split(":")[1].strip(); break
    except Exception: pass
    return info
ALL["ortam"] = environment()
for k, v in ALL["ortam"].items(): print(f"{k:<14}: {v}")

# ---------------- 1. VERİ VE BÖLME PROTOKOLÜ (Hakem #2 Yorum #2, #19-V) ----------------
# Kaynak: UCI ML Repository, Breast Cancer Wisconsin (Diagnostic) (Wolberg, Street ve Mangasarian, 1995);
# scikit-learn load_breast_cancer kopyası. 569 örnek, 30 öznitelik (ince iğne aspirasyonu görüntülerinden
# hesaplanmış hücre çekirdeği morfolojik ölçümleri), 212 malign / 357 benign. Alt-örnekleme YAPILMAZ.
RANDOM_SEED = 42; N_QUBITS = 4; N_SPLITS = 5
np.random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)
bc = load_breast_cancer()
X_raw = bc.data.astype(float)
y = (bc.target == 0).astype(int)          # 1 = Malign (pozitif sınıf), 0 = Benign
CLASS_NAMES = ["Benign", "Malign"]
print(f"\nToplam örnek: {len(y)} | Benign: {np.sum(y==0)} | Malign: {np.sum(y==1)} | Öznitelik: {X_raw.shape[1]}")

FOLDS = list(StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED).split(X_raw, y))
json.dump({f"fold{k}": {"train": tr.tolist(), "test": te.tolist()} for k, (tr, te) in enumerate(FOLDS, 1)},
          open(f"{OUT}/bc_cv_fold_indices.json", "w"))
for k, (tr, te) in enumerate(FOLDS, 1):
    print(f"Fold {k}: eğitim {len(tr)} (M={y[tr].sum()}) | test {len(te)} (M={y[te].sum()})")

def angle_pipeline(X_tr, X_te, n_comp=N_QUBITS):
    """StandardScaler -> PCA(4) -> MinMax[-pi, pi]; YALNIZCA eğitim katmanında fit (Hakem #2 Yorum #3)."""
    sc = StandardScaler().fit(X_tr)
    pca = PCA(n_components=n_comp, random_state=RANDOM_SEED).fit(sc.transform(X_tr))
    mm = MinMaxScaler(feature_range=(-np.pi, np.pi)).fit(pca.transform(sc.transform(X_tr)))
    f = lambda Z: mm.transform(pca.transform(sc.transform(Z)))
    return f(X_tr), f(X_te), pca.explained_variance_ratio_

ANGLES = [angle_pipeline(X_raw[tr], X_raw[te]) for tr, te in FOLDS]
pca_var = np.array([a[2] for a in ANGLES])
ALL["pca_varyans_katmanlar"] = pca_var.tolist()
print("\nPCA bileşen varyansları (katman ort., %):", np.round(pca_var.mean(0) * 100, 2),
      f"| kümülatif: %{ms(pca_var.sum(1), 100)}")


In [ ]:
# ---------------- 2. KUANTUM SADAKAT MATRİSLERİ (her katmanda bir kez; SVM, k-NN ve kümelemede ortak kullanılır) ----------------
# Analitik beklenti değeri (shots=None). Adjoint: 4 kübit, |<0|U†(x')U(x)|0>|^2. Swap Test: 9 kübit, <Z_0>.
dev = qml.device("default.qubit", wires=N_QUBITS)
@qml.qnode(dev, interface="autograd")
def adjoint_qnode(x1, x2):
    qml.AngleEmbedding(x1, wires=range(N_QUBITS))
    qml.adjoint(qml.AngleEmbedding)(x2, wires=range(N_QUBITS))
    return qml.probs(wires=range(N_QUBITS))

dev_swap = qml.device("default.qubit", wires=2 * N_QUBITS + 1)
@qml.qnode(dev_swap, interface="autograd")
def swap_test_qnode(x1, x2):
    qml.Hadamard(wires=0)
    qml.AngleEmbedding(x1, wires=range(1, N_QUBITS + 1))
    qml.AngleEmbedding(x2, wires=range(N_QUBITS + 1, 2 * N_QUBITS + 1))
    for i in range(N_QUBITS):
        qml.CSWAP(wires=[0, i + 1, i + N_QUBITS + 1])
    qml.Hadamard(wires=0)
    return qml.expval(qml.PauliZ(0))

def fidelity(kind, a, b):
    return float(adjoint_qnode(a, b)[0]) if kind == "adjoint" else float(np.clip(swap_test_qnode(a, b), 0.0, 1.0))

def fidelity_matrix(kind, A, B=None):
    sym = B is None; B = A if sym else B
    F = np.zeros((len(A), len(B)))
    for i in range(len(A)):
        for j in range(len(B)):
            if sym and j < i: F[i, j] = F[j, i]
            elif sym and i == j: F[i, j] = 1.0
            else: F[i, j] = fidelity(kind, A[i], B[j])
    return F

KER = {}   # KER[(fold, kind)] = {"tr", "te", "t_tr", "t_te"}
t_all = time.time()
for k, (A_tr, A_te, _) in enumerate(ANGLES, 1):
    for kind in ("adjoint", "swap"):
        t0 = time.time(); F_tr = fidelity_matrix(kind, A_tr); t1 = time.time() - t0
        t0 = time.time(); F_te = fidelity_matrix(kind, A_te, A_tr); t2 = time.time() - t0
        KER[(k, kind)] = {"tr": F_tr, "te": F_te, "t_tr": t1, "t_te": t2}
        print(f"Fold {k} {kind:<7}: eğitim matrisi {t1:6.1f} sn | test matrisi {t2:6.1f} sn | geçen toplam {(time.time()-t_all)/60:5.1f} dk")
diff = max(np.abs(KER[(k, "adjoint")]["tr"] - KER[(k, "swap")]["tr"]).max() for k in range(1, 6))
print(f"\nAdjoint ve Swap Test sadakatleri arasındaki en büyük fark: {diff:.2e}")
ALL["kernel_sureleri"] = {f"fold{k}_{kind}": {"egitim_sn": v["t_tr"], "test_sn": v["t_te"]} for (k, kind), v in KER.items()}
ALL["adjoint_swap_max_fark"] = float(diff)
kernel_time = lambda k, kind: KER[(k, kind)]["t_tr"] + KER[(k, kind)]["t_te"]
qdist = lambda F: np.sqrt(np.maximum(0.0, 2.0 - 2.0 * np.clip(F, 0.0, 1.0)))   # D = sqrt(2 - 2F)


In [ ]:
# ---------------- 3. KLASİK SVM / QSVM ve KLASİK k-NN / QKNN ----------------
# Süre (katman başına): klasik modeller için eğitim+test; kuantum modeller için o katmanın sadakat matrisi
# hesaplama süresi + eğitim+test. Sadakat matrisleri QSVM, QKNN ve kuantum çekirdek kümeleme arasında ortaktır.
K_NEIGHBORS = 5
def binary_metrics(y_true, pred, score):
    return {"acc": accuracy_score(y_true, pred), "f1": f1_score(y_true, pred, zero_division=0),
            "auc": roc_auc_score(y_true, score), "cm": confusion_matrix(y_true, pred, labels=[0, 1])}

SUP_MODELS = ["Klasik SVM (Linear)", "Klasik SVM (RBF)", "QSVM (Adjoint Kernel)", "QSVM (Swap Test)",
              "Klasik k-NN (Öklid)", "QKNN (Adjoint Kernel)", "QKNN (Swap Test)"]
SUP = {m: {"acc": [], "f1": [], "auc": [], "time": [], "cm": np.zeros((2, 2), int)} for m in SUP_MODELS}
for k, ((tr, te), (A_tr, A_te, _)) in enumerate(zip(FOLDS, ANGLES), 1):
    y_tr, y_te = y[tr], y[te]
    for m in SUP_MODELS:
        kind = "adjoint" if "Adjoint" in m else "swap"
        t0 = time.time()
        if m.startswith("Klasik SVM"):
            clf = SVC(kernel="linear" if "Linear" in m else "rbf", C=1.0, probability=True, random_state=RANDOM_SEED).fit(A_tr, y_tr)
            pred, score = clf.predict(A_te), clf.predict_proba(A_te)[:, 1]
        elif m.startswith("QSVM"):
            clf = SVC(kernel="precomputed", C=1.0, probability=True, random_state=RANDOM_SEED).fit(KER[(k, kind)]["tr"], y_tr)
            pred, score = clf.predict(KER[(k, kind)]["te"]), clf.predict_proba(KER[(k, kind)]["te"])[:, 1]
        elif m.startswith("Klasik k-NN"):
            clf = KNeighborsClassifier(n_neighbors=K_NEIGHBORS, metric="euclidean").fit(A_tr, y_tr)
            pred, score = clf.predict(A_te), clf.predict_proba(A_te)[:, 1]
        else:
            D_tr, D_te = qdist(KER[(k, kind)]["tr"]), qdist(KER[(k, kind)]["te"]); np.fill_diagonal(D_tr, 0.0)
            clf = KNeighborsClassifier(n_neighbors=K_NEIGHBORS, metric="precomputed").fit(D_tr, y_tr)
            pred, score = clf.predict(D_te), clf.predict_proba(D_te)[:, 1]
        el = time.time() - t0 + (0.0 if m.startswith("Klasik") else kernel_time(k, kind))
        r = binary_metrics(y_te, pred, score); s = SUP[m]
        for key in ("acc", "f1", "auc"): s[key].append(r[key])
        s["time"].append(el); s["cm"] += r["cm"]

def print_sup(title, models):
    print("\n" + "=" * 112); print(title); print("=" * 112)
    print(f"{'Model':<24} | {'Doğruluk (%)':<15} | {'F1 Malign (%)':<15} | {'ROC-AUC':<15} | {'Süre/katman (sn)'}"); print("-" * 112)
    for m in models:
        s = SUP[m]; print(f"{m:<24} | {ms(s['acc'],100):<15} | {ms(s['f1'],100):<15} | {ms(s['auc'],1,3):<15} | {ms(s['time'],1,3)}")
    print("=" * 112)
print_sup("BREAST CANCER – SVM / QSVM (5-katlı CV, Mean ± SD)", SUP_MODELS[:4])
print_sup("BREAST CANCER – k-NN / QKNN (k=5, 5-katlı CV, Mean ± SD)", SUP_MODELS[4:])
ALL["svm_knn"] = {m: {k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in s.items()} for m, s in SUP.items()}

fig, axes = plt.subplots(1, 7, figsize=(24, 3.6), dpi=200)
for ax, m in zip(axes, SUP_MODELS):
    sns.heatmap(SUP[m]["cm"], annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False, xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    ax.set_title(f"{m}\n(5 katman toplamı)", fontsize=9); ax.set_xlabel("Tahmin"); ax.set_ylabel("Gerçek")
plt.tight_layout(); plt.savefig(f"{OUT}/bc_svm_knn_karmasiklik.png", dpi=300); plt.show()


In [ ]:
# ---------------- 4. KÜMELEME (k=2): Klasik K-Means, Kuantum Çekirdek Kümeleme, Swap-Test Q-Means ----------------
# Kümeler test katmanında Hungarian (Kuhn-Munkres) optimal ataması ile sınıflara eşlenir (Hakem #2 Yorum #13).
# Silüet: test katmanının açı uzayında (Öklid); ARI ve NMI eşlemeden bağımsızdır.
N_CLUSTERS = 2
def cluster_eval(y_true, clusters, X_eval):
    cm = confusion_matrix(y_true, clusters, labels=range(N_CLUSTERS))
    r, c = linear_sum_assignment(cm.max() - cm); mp = {cc: rr for rr, cc in zip(r, c)}
    ya = np.array([mp[v] for v in clusters])
    return {"acc": accuracy_score(y_true, ya), "f1": f1_score(y_true, ya, zero_division=0),
            "ari": adjusted_rand_score(y_true, clusters), "nmi": normalized_mutual_info_score(y_true, clusters),
            "sil": silhouette_score(X_eval, clusters) if len(set(clusters)) > 1 else 0.0}

class QuantumKMeans:
    """9 kübitlik Swap-Test mesafesiyle iteratif merkez güncellemesi (Fraud koduyla aynı)."""
    def __init__(self, n_clusters=2, max_iter=5, random_state=RANDOM_SEED):
        self.k, self.max_iter, self.rs = n_clusters, max_iter, random_state
    def _d(self, x, c): return float(qdist(np.clip(swap_test_qnode(x, c), 0, 1)))
    def fit(self, X):
        rng = np.random.RandomState(self.rs)
        self.centroids = X[rng.choice(len(X), self.k, replace=False)].copy(); labels = np.full(len(X), -1)
        for _ in range(self.max_iter):
            new = np.array([np.argmin([self._d(x, c) for c in self.centroids]) for x in X])
            if np.all(new == labels): break
            labels = new
            for j in range(self.k):
                if np.any(labels == j): self.centroids[j] = X[labels == j].mean(axis=0)
        return self
    def predict(self, X): return np.array([np.argmin([self._d(x, c) for c in self.centroids]) for x in X])

CL_MODELS = ["Klasik K-Means", "Kuantum Çekirdek Kümeleme", "Swap Test K-Means"]
CL = {m: {k: [] for k in ("acc", "f1", "ari", "nmi", "sil", "time")} for m in CL_MODELS}
for k, ((tr, te), (A_tr, A_te, _)) in enumerate(zip(FOLDS, ANGLES), 1):
    y_te = y[te]
    t0 = time.time(); c = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_SEED, n_init=10).fit(A_tr).predict(A_te)
    r = cluster_eval(y_te, c, A_te); r["time"] = time.time() - t0
    for key in r: CL["Klasik K-Means"][key].append(r[key])
    t0 = time.time()
    D_tr, D_te = qdist(KER[(k, "adjoint")]["tr"]), qdist(KER[(k, "adjoint")]["te"]); np.fill_diagonal(D_tr, 0.0)
    kq = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_SEED, n_init=10).fit(D_tr)
    medoids = [int(np.argmin(np.linalg.norm(D_tr - cc, axis=1))) for cc in kq.cluster_centers_]
    c = np.argmin(D_te[:, medoids], axis=1)
    r = cluster_eval(y_te, c, A_te); r["time"] = time.time() - t0 + kernel_time(k, "adjoint")
    for key in r: CL["Kuantum Çekirdek Kümeleme"][key].append(r[key])
    t0 = time.time(); c = QuantumKMeans().fit(A_tr).predict(A_te)
    r = cluster_eval(y_te, c, A_te); r["time"] = time.time() - t0
    for key in r: CL["Swap Test K-Means"][key].append(r[key])
    print(f"Fold {k}/5 tamamlandı")

print("\n" + "=" * 136); print("BREAST CANCER – KÜMELEME (k=2, 5-katlı CV, Mean ± SD, Hungarian eşleme)"); print("=" * 136)
print(f"{'Model':<26} | {'Doğruluk (%)':<15} | {'F1 Malign (%)':<15} | {'ARI':<14} | {'NMI':<14} | {'Silüet':<14} | {'Süre/katman (sn)'}"); print("-" * 136)
for m in CL_MODELS:
    s = CL[m]; print(f"{m:<26} | {ms(s['acc'],100):<15} | {ms(s['f1'],100):<15} | {ms(s['ari'],1,3):<14} | {ms(s['nmi'],1,3):<14} | {ms(s['sil'],1,3):<14} | {ms(s['time'],1,3)}")
print("=" * 136)
ALL["kumeleme"] = CL


In [ ]:
# ---------------- 5. SAF KLASİK ANN, KLASİK KONTROL ve HİBRİT QNN (Hakem #2 Yorum #5, #8, #11) ----------------
# Eşit koşullar: aynı 5 katman, Adam (lr=0.002), batch=32, PyTorch varsayılan başlatma (tohum = 42 + katman no),
# erken durdurma YOK, tüm modellerde 200 epoch'luk aynı bütçe. 30/100/200 sonuçları aynı eğitimin kontrol noktalarıdır.
# Çıktı: tek logit + BCEWithLogitsLoss (Fraud ile aynı). Girdi: 30 öznitelik, MinMax[-1,1] yalnızca eğitimde fit.
LR, BATCH_SIZE, MAX_EPOCHS, CHECKPOINTS = 0.002, 32, 200, (30, 100, 200)
INPUT_DIM = X_raw.shape[1]

def make_qlayer(n_qubits, n_layers=2):
    qdev = qml.device("default.qubit", wires=n_qubits)
    @qml.qnode(qdev, interface="torch", diff_method="backprop")
    def qnode(inputs, weights):
        qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="Y")
        qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
        return [qml.expval(qml.PauliZ(wires=i)) for i in range(n_qubits)]
    return qml.qnn.TorchLayer(qnode, {"weights": (n_layers, n_qubits, 3)})

class HybridQNN(nn.Module):
    def __init__(self, n_qubits):
        super().__init__()
        self.fc1 = nn.Linear(INPUT_DIM, 16); self.dropout = nn.Dropout(0.3)
        self.fc_bottleneck = nn.Linear(16, n_qubits); self.qlayer = make_qlayer(n_qubits)
        self.fc_out = nn.Linear(n_qubits, 1)
    def forward(self, x):
        x = self.dropout(torch.relu(self.fc1(x)))
        x = torch.tanh(torch.relu(self.fc_bottleneck(x))) * np.pi
        return self.fc_out(self.qlayer(x))

class ClassicalControl(nn.Module):
    """Kuantum katman yerine Linear(4,4)+tanh; diğer her şey QNN-4 ile aynı (yaklaşık eşit parametre sayısı)."""
    def __init__(self, n_bottleneck=4):
        super().__init__()
        self.fc1 = nn.Linear(INPUT_DIM, 16); self.dropout = nn.Dropout(0.3)
        self.fc_bottleneck = nn.Linear(16, n_bottleneck); self.classical_layer = nn.Linear(n_bottleneck, n_bottleneck)
        self.fc_out = nn.Linear(n_bottleneck, 1)
    def forward(self, x):
        x = self.dropout(torch.relu(self.fc1(x)))
        x = torch.tanh(torch.relu(self.fc_bottleneck(x))) * np.pi
        return self.fc_out(torch.tanh(self.classical_layer(x)))

class PureClassicalANN(nn.Module):
    """Fraud deneyleriyle aynı saf klasik ANN: 30 -> 16 -> Dropout -> 2 -> 2 (ReLU) -> 1."""
    def __init__(self, bottleneck_dim=2):
        super().__init__()
        self.fc1 = nn.Linear(INPUT_DIM, 16); self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(16, bottleneck_dim); self.fc3 = nn.Linear(bottleneck_dim, bottleneck_dim)
        self.fc_out = nn.Linear(bottleneck_dim, 1)
    def bottleneck(self, x):
        return torch.relu(self.fc3(torch.relu(self.fc2(self.dropout(torch.relu(self.fc1(x)))))))
    def forward(self, x): return self.fc_out(self.bottleneck(x))

def count_params(model):
    tot = sum(p.numel() for p in model.parameters() if p.requires_grad)
    q = sum(p.numel() for n, p in model.named_parameters() if p.requires_grad and n.startswith("qlayer"))
    return tot, q

def train_eval(builder, tr, te, seed, lr=LR, max_epochs=MAX_EPOCHS, checkpoints=CHECKPOINTS):
    torch.manual_seed(seed); np.random.seed(seed)
    sc = MinMaxScaler(feature_range=(-1, 1)).fit(X_raw[tr])
    X_tr = torch.tensor(sc.transform(X_raw[tr]), dtype=torch.float32); X_te = torch.tensor(sc.transform(X_raw[te]), dtype=torch.float32)
    y_tr = torch.tensor(y[tr], dtype=torch.float32).unsqueeze(1); y_te = y[te]
    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
    model = builder(); crit = nn.BCEWithLogitsLoss(); opt = optim.Adam(model.parameters(), lr=lr)
    curve = {"train_loss": [], "val_loss": [], "val_acc": []}; ck = {}; t_train = 0.0
    for ep in range(1, max_epochs + 1):
        t0 = time.time(); model.train(); losses = []
        for bx, by in loader:
            opt.zero_grad(); loss = crit(model(bx), by); loss.backward(); opt.step(); losses.append(loss.item())
        t_train += time.time() - t0
        model.eval()
        with torch.no_grad():
            logit = model(X_te); prob = torch.sigmoid(logit).numpy().ravel()
            vloss = crit(logit, torch.tensor(y_te, dtype=torch.float32).unsqueeze(1)).item()
        pred = (prob > 0.5).astype(int)
        curve["train_loss"].append(float(np.mean(losses))); curve["val_loss"].append(vloss); curve["val_acc"].append(accuracy_score(y_te, pred))
        if ep in checkpoints:
            entry = {"acc": accuracy_score(y_te, pred), "f1": f1_score(y_te, pred, zero_division=0),
                     "auc": roc_auc_score(y_te, prob) if len(np.unique(prob)) > 1 else 0.5, "train_time": t_train,
                     "tek_sinif_tahmini": bool(len(np.unique(pred)) == 1), "cm": confusion_matrix(y_te, pred, labels=[0, 1]).tolist()}
            if isinstance(model, PureClassicalANN):
                with torch.no_grad(): h = model.bottleneck(X_tr).numpy()
                entry["olu_relu_birim"] = int(np.sum(h.max(axis=0) <= 0))   # tüm eğitim örneklerinde 0 çıkan darboğaz nöronu
            ck[ep] = entry
    return ck, curve

NN_MODELS = {"Saf Klasik ANN": lambda: PureClassicalANN(2), "Klasik Kontrol (4 boyutlu darboğaz)": lambda: ClassicalControl(4),
             "Hibrit QNN (2 Kübit)": lambda: HybridQNN(2), "Hibrit QNN (4 Kübit)": lambda: HybridQNN(4), "Hibrit QNN (8 Kübit)": lambda: HybridQNN(8)}
NN = {}
for name, builder in NN_MODELS.items():
    t0 = time.time(); runs = [train_eval(builder, tr, te, RANDOM_SEED + k) for k, (tr, te) in enumerate(FOLDS, 1)]
    tot, q = count_params(builder())
    NN[name] = {"params_total": tot, "params_quantum": q, "curves": [c for _, c in runs],
                "checkpoints": {ep: [ck[ep] for ck, _ in runs] for ep in CHECKPOINTS}}
    print(f"[{name}] tamamlandı ({(time.time()-t0)/60:.1f} dk) -> " + " | ".join(f"Ep{ep}: %{np.mean([r['acc'] for r in NN[name]['checkpoints'][ep]])*100:.2f}" for ep in CHECKPOINTS))

print("\n" + "=" * 152); print("BREAST CANCER – ANN / KLASİK KONTROL / QNN (5-katlı CV, Mean ± SD, süre: katman başına eğitim süresi)"); print("=" * 152)
print(f"{'Model':<36} | {'Param (top/kuantum)':<19} | {'Epoch':<5} | {'Doğruluk (%)':<15} | {'F1 Malign (%)':<15} | {'ROC-AUC':<14} | {'Süre (sn)':<14} | Tek sınıf"); print("-" * 152)
for name, s in NN.items():
    for ep in CHECKPOINTS:
        r = s["checkpoints"][ep]; g = lambda key: [x[key] for x in r]
        print(f"{name:<36} | {str(s['params_total'])+' / '+str(s['params_quantum']):<19} | {ep:<5} | {ms(g('acc'),100):<15} | {ms(g('f1'),100):<15} | {ms(g('auc'),1,3):<14} | {ms(g('train_time'),1,1):<14} | {sum(g('tek_sinif_tahmini'))}/5")
print("=" * 152)
ALL["ann_qnn"] = {n: {"params_total": s["params_total"], "params_quantum": s["params_quantum"], "curves": s["curves"],
                      "checkpoints": {str(ep): v for ep, v in s["checkpoints"].items()}} for n, s in NN.items()}

# Öğrenme eğrileri (Hakem #2 Yorum #5, #19-VI)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), dpi=200); ep_ax = np.arange(1, MAX_EPOCHS + 1)
for name, s in NN.items():
    for ax, key, sc_ in ((axes[0], "val_acc", 100), (axes[1], "val_loss", 1)):
        arr = np.array([c[key] for c in s["curves"]]) * sc_; m_, sd = arr.mean(0), arr.std(0)
        ax.plot(ep_ax, m_, label=name, linewidth=1.2); ax.fill_between(ep_ax, m_ - sd, m_ + sd, alpha=0.12)
for ax, yl in ((axes[0], "Doğrulama doğruluğu (%)"), (axes[1], "Doğrulama kaybı (BCE)")):
    ax.set_xlabel("Epoch"); ax.set_ylabel(yl); ax.grid(alpha=0.3)
    for c in CHECKPOINTS: ax.axvline(c, color="gray", linestyle=":", linewidth=0.8)
axes[0].legend(fontsize=8, loc="lower right"); fig.suptitle("Breast Cancer: 5 katman ortalaması ± SD öğrenme eğrileri")
plt.tight_layout(); plt.savefig(f"{OUT}/bc_ann_qnn_ogrenme_egrileri.png", dpi=300); plt.show()

# 200. epoch karmaşıklık matrisleri (5 katman toplamı)
fig, axes = plt.subplots(1, 5, figsize=(20, 3.6), dpi=200)
for ax, (name, s) in zip(axes, NN.items()):
    cm = np.sum([np.array(r["cm"]) for r in s["checkpoints"][200]], axis=0)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Purples", ax=ax, cbar=False, xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    ax.set_title(f"{name}\n200 epoch", fontsize=9); ax.set_xlabel("Tahmin"); ax.set_ylabel("Gerçek")
plt.tight_layout(); plt.savefig(f"{OUT}/bc_ann_qnn_karmasiklik_200ep.png", dpi=300); plt.show()


In [ ]:
# ---------------- 6. KLASİK ANN ÇÖKME ANALİZİ: ÇOKLU TOHUM VE ÖĞRENME ORANI (Hakem #2 Yorum #6) ----------------
# Aynı 5 katman; 5 farklı tohum (başlatma, dropout maskesi ve mini-batch sırası değişir) x 3 öğrenme oranı.
# Her koşum için 30/100/200. epoch sonuçları, tek sınıf tahmini (çökme) ve darboğazdaki ölü ReLU nöron sayısı kaydedilir.
# Karşılaştırma: aynı tohumlarla Klasik Kontrol modeli (ReLU-2 darboğaz yerine tanh-4 darboğaz), lr=0.002.
SEEDS = [0, 1, 2, 3, 4]; LRS = [0.0005, 0.002, 0.01]
MS = {}
t_ms = time.time()
for lr in LRS:
    runs = []
    for sd in SEEDS:
        for k, (tr, te) in enumerate(FOLDS, 1):
            ck, curve = train_eval(lambda: PureClassicalANN(2), tr, te, seed=1000 * sd + k, lr=lr)
            runs.append({"seed": sd, "fold": k, "ck": ck, "val_acc": curve["val_acc"], "val_loss": curve["val_loss"]})
    MS[f"ANN lr={lr}"] = runs
    print(f"ANN lr={lr} tamamlandı ({(time.time()-t_ms)/60:.1f} dk)")
runs = []
for sd in SEEDS:
    for k, (tr, te) in enumerate(FOLDS, 1):
        ck, curve = train_eval(lambda: ClassicalControl(4), tr, te, seed=1000 * sd + k, lr=0.002)
        runs.append({"seed": sd, "fold": k, "ck": ck, "val_acc": curve["val_acc"], "val_loss": curve["val_loss"]})
MS["Klasik Kontrol lr=0.002"] = runs

print("\n" + "=" * 128); print("BREAST CANCER – KLASİK ANN ÇOKLU TOHUM ANALİZİ (5 tohum x 5 katman = 25 koşum / ayar)"); print("=" * 128)
print(f"{'Ayar':<26} | {'Epoch':<5} | {'Doğruluk (%)':<15} | {'F1 Malign (%)':<15} | {'Çöken koşum':<12} | {'Ölü ReLU (ort.)':<15} | Tohum bazlı doğruluk ort. (%)"); print("-" * 128)
for name, runs in MS.items():
    for ep in CHECKPOINTS:
        acc = [r["ck"][ep]["acc"] for r in runs]; f1 = [r["ck"][ep]["f1"] for r in runs]
        col = sum(r["ck"][ep]["tek_sinif_tahmini"] for r in runs)
        dead = [r["ck"][ep].get("olu_relu_birim", np.nan) for r in runs]
        per_seed = [np.mean([r["ck"][ep]["acc"] for r in runs if r["seed"] == sd]) * 100 for sd in SEEDS]
        print(f"{name:<26} | {ep:<5} | {ms(acc,100):<15} | {ms(f1,100):<15} | {str(col)+'/25':<12} | {('-' if np.all(np.isnan(dead)) else f'{np.nanmean(dead):.2f}'):<15} | {', '.join(f'{v:.1f}' for v in per_seed)}")
print("=" * 128)
ALL["ann_coklu_tohum"] = {n: [{"seed": r["seed"], "fold": r["fold"], "ck": {str(e): v for e, v in r["ck"].items()},
                               "val_acc": r["val_acc"], "val_loss": r["val_loss"]} for r in runs] for n, runs in MS.items()}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), dpi=200, sharey=True)
for ax, name in zip(axes, ["ANN lr=0.002", "Klasik Kontrol lr=0.002"]):
    for r in MS[name]:
        ax.plot(np.arange(1, MAX_EPOCHS + 1), np.array(r["val_acc"]) * 100, linewidth=0.7,
                color="crimson" if r["ck"][200]["tek_sinif_tahmini"] else "steelblue", alpha=0.6)
    ax.set_title(f"{name}: 25 koşum (kırmızı: 200. epoch'ta tek sınıf tahmini)", fontsize=9)
    ax.set_xlabel("Epoch"); ax.grid(alpha=0.3)
axes[0].set_ylabel("Doğrulama doğruluğu (%)")
plt.tight_layout(); plt.savefig(f"{OUT}/bc_ann_coklu_tohum.png", dpi=300); plt.show()


In [ ]:
# ---------------- 7. ANOMALİ TESPİTİ: KLASİK AE ve KUANTUM QAE (Hakem #2 Yorum #12) ----------------
# Normal sınıf: Benign (357 örneğin tamamı). Anomali sınıfı: Malign örneklerden rastgele 89 örnek (random_state=42).
# Toplam 446 örnek, %80 normal / %20 anomali (Fraud anomali deneyindeki 400/100 oranı ile aynı).
# Tabakalı 5-katlı CV; her katmanda modeller YALNIZCA eğitim katmanındaki normal örneklerle eğitilir ve ön-işleme
# (StandardScaler -> PCA(4) -> MinMax[-pi,pi]) yalnızca bu normal örneklerle fit edilir.
# Anomali skoru: Klasik AE için yeniden yapılandırma MSE'si; QAE için 1 - (<Z2>+<Z3>+2)/4 (çöp kübit sapması).
# Karar eşiği: eğitim katmanındaki normal örneklerin skorlarının 95. persentili (test etiketleri KULLANILMAZ).
# QAE eğitilebilirdir: 2 katmanlı StronglyEntanglingLayers ağırlıkları (2 x 4 x 3 = 24 parametre) optimize edilir.
rng = np.random.RandomState(RANDOM_SEED)
idx_norm = np.where(y == 0)[0]; idx_anom = rng.choice(np.where(y == 1)[0], 89, replace=False)
idx_ad = np.concatenate([idx_norm, idx_anom]); X_ad = X_raw[idx_ad]
y_ad = np.r_[np.zeros(len(idx_norm), int), np.ones(len(idx_anom), int)]
FOLDS_AD = list(StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED).split(X_ad, y_ad))
print(f"Anomali örneklemi: {len(y_ad)} (Normal {np.sum(y_ad==0)}, Anomali {np.sum(y_ad==1)})")
for k, (tr, te) in enumerate(FOLDS_AD, 1):
    print(f"Fold {k}: eğitimde kullanılan normal {np.sum(y_ad[tr]==0)} | test {len(te)} (normal {np.sum(y_ad[te]==0)}, anomali {np.sum(y_ad[te]==1)})")
AE_CK, AE_MAX = (20, 30, 50, 100), 100

class ClassicalAE(nn.Module):
    def __init__(self, deep):
        super().__init__()
        self.net = (nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2), nn.ReLU(), nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 4))
                    if deep else nn.Sequential(nn.Linear(4, 2), nn.ReLU(), nn.Linear(2, 4)))
    def forward(self, x): return self.net(x)
    def loss(self, x): return nn.functional.mse_loss(self(x), x)
    def score(self, x):
        with torch.no_grad(): return torch.mean((x - self(x)) ** 2, dim=1).numpy()

qae_dev = qml.device("default.qubit", wires=N_QUBITS)
@qml.qnode(qae_dev, interface="torch", diff_method="backprop")
def qae_qnode(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(N_QUBITS))
    qml.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
    return [qml.expval(qml.PauliZ(2)), qml.expval(qml.PauliZ(3))]

class QuantumAE(nn.Module):
    def __init__(self):
        super().__init__(); self.qlayer = qml.qnn.TorchLayer(qae_qnode, {"weights": (2, N_QUBITS, 3)})
    def _s(self, x):
        z = self.qlayer(x); return 1.0 - (z[:, 0] + z[:, 1] + 2.0) / 4.0
    def loss(self, x): return torch.mean(self._s(x))
    def score(self, x):
        with torch.no_grad(): return self._s(x).numpy()

AE_MODELS = {"Klasik AE (Sığ 4-2-4)": (lambda: ClassicalAE(False), 0.005),
             "Klasik AE (Derin 4-8-2-8-4)": (lambda: ClassicalAE(True), 0.005),
             "Kuantum QAE (2 Katman)": (lambda: QuantumAE(), 0.02)}
AE = {m: {ep: {k: [] for k in ("auc", "pr_auc", "acc", "f1", "train_time")} for ep in AE_CK} for m in AE_MODELS}
for name, (builder, lr) in AE_MODELS.items():
    t_m = time.time()
    for k, (tr, te) in enumerate(FOLDS_AD, 1):
        torch.manual_seed(RANDOM_SEED + k)
        tr_n = tr[y_ad[tr] == 0]
        A_trn, A_te, _ = angle_pipeline(X_ad[tr_n], X_ad[te])
        x_trn = torch.tensor(A_trn, dtype=torch.float32); x_te = torch.tensor(A_te, dtype=torch.float32)
        loader = DataLoader(x_trn, batch_size=32, shuffle=True)
        model = builder(); opt = optim.Adam(model.parameters(), lr=lr); t_train = 0.0
        for ep in range(1, AE_MAX + 1):
            t0 = time.time(); model.train()
            for bx in loader:
                opt.zero_grad(); loss = model.loss(bx); loss.backward(); opt.step()
            t_train += time.time() - t0
            if ep in AE_CK:
                model.eval(); thr = np.percentile(model.score(x_trn), 95)
                s_te = model.score(x_te); pred = (s_te > thr).astype(int); yt = y_ad[te]; r = AE[name][ep]
                r["auc"].append(roc_auc_score(yt, s_te)); r["pr_auc"].append(average_precision_score(yt, s_te))
                r["acc"].append(accuracy_score(yt, pred)); r["f1"].append(f1_score(yt, pred, zero_division=0)); r["train_time"].append(t_train)
    print(f"[{name}] tamamlandı ({(time.time()-t_m)/60:.1f} dk)")

print("\n" + "=" * 136); print("BREAST CANCER – ANOMALİ TESPİTİ (5-katlı CV, Mean ± SD, pozitif sınıf: anomali/malign, eşik: eğitim normal 95. persentil)"); print("=" * 136)
print(f"{'Model':<30} | {'Epoch':<5} | {'ROC-AUC (%)':<15} | {'PR-AUC (%)':<15} | {'Doğruluk (%)':<15} | {'F1 (%)':<15} | {'Süre/katman (sn)'}"); print("-" * 136)
for name in AE_MODELS:
    for ep in AE_CK:
        r = AE[name][ep]
        print(f"{name:<30} | {ep:<5} | {ms(r['auc'],100):<15} | {ms(r['pr_auc'],100):<15} | {ms(r['acc'],100):<15} | {ms(r['f1'],100):<15} | {ms(r['train_time'],1,2)}")
print("=" * 136)
ALL["anomali"] = {"anomali_indeksleri": idx_anom.tolist(), "sonuclar": {n: {str(e): v for e, v in r.items()} for n, r in AE.items()}}


In [ ]:
# ---------------- 8. TÜM SONUÇLARI KAYDET (Hakem #2 Yorum #18) ----------------
def to_py(o):
    if isinstance(o, dict): return {str(k): to_py(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [to_py(v) for v in o]
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    return o
json.dump(to_py(ALL), open(f"{OUT}/bc_tum_sonuclar.json", "w"), ensure_ascii=False)
import shutil; shutil.make_archive("bc_sonuclar", "zip", OUT)
print("Kaydedildi: bc_sonuclar.zip (JSON + grafikler + bölme indeksleri). Bu dosyayı indirip gönderin.")
try:
    from google.colab import files; files.download("bc_sonuclar.zip")
except Exception: pass
